In [2]:
from collections import defaultdict
from typing import Optional

import numpy as np
import torch
import tqdm
from tensordict import TensorDict, TensorDictBase
from tensordict.nn import TensorDictModule
from torch import nn

from torchrl.data import BoundedTensorSpec, CompositeSpec, UnboundedContinuousTensorSpec, DiscreteTensorSpec
from torchrl.envs import (
    CatTensors,
    EnvBase,
    Transform,
    TransformedEnv,
    UnsqueezeTransform,
)
from torchrl.envs.transforms.transforms import _apply_to_composite
from torchrl.envs.utils import check_env_specs, step_mdp

In [3]:

class MomentumEnv(EnvBase):

    def __init__(self, batch_size=(), device=None, **kwargs):

        self.n_agents = 6
    
        self.REFRESH_RATE = 240
        self.dt = 1 / self.REFRESH_RATE

        # Team 1
        self.player1 = Player([1000, 1000], [0, -1], 0, Shotgun(self.REFRESH_RATE))
        self.player2 = Player([2000, 1000], [0, -1], 0, AssaultRifle(self.REFRESH_RATE)
        )
        self.player3 = Player([3000, 1000], [0, -1], 0, SniperRifle(self.REFRESH_RATE)
        )

        # Team 2
        self.player4 = Player([3000, 3000], [0, 1], 1, Shotgun(self.REFRESH_RATE))
        self.player5 = Player([2000, 3000], [0, 1], 1, AssaultRifle(self.REFRESH_RATE))
        self.player6 = Player([1000, 3000], [0, 1], 1, SniperRifle(self.REFRESH_RATE))

        # Contains all of the players in the game
        self.players = [self.player1, self.player2, self.player3, self.player4, self.player5, self.player6]

        #Blocks
        self.block1 = Block([1500, 2000], 200, 1200)
        self.block2 = Block([2500, 2000], 200, 1200)
        self.block3 = Block([1000, 2000], 400, 400)
        self.block4 = Block([2000, 2000], 400, 400)
        self.block5 = Block([3000, 2000], 400, 400)

        #List of all blocks
        self.blocks = [self.block1, self.block2, self.block3, self.block4, self.block5]

        #Set countdowns
        self.countdowns = {p: 0 for p in self.players}

        #Player strings
        self.possible_agents = ["player1", "player2", "player3", "player4", "player5", "player6"]

        #Current timestep
        self.timesteps = 0

        #Max timesteps in a game
        self.max_timestep = 6000

        self.scores = {0: 0, 
                       1: 0}

        self.max_score = 600

    
    def _step(self, tensordict):

        rays = {}
        #Update player states
        for i, player in enumerate(self.players, 0):

            playerStringLabel = self.invPlayerLabels[player]
            playerActions = actions[playerStringLabel]

            action = {"forward": playerActions[1],
                    "right": playerActions[2],
                    "rot_rate": playerActions[3],
                    "firing": playerActions[0]}

            rays[player], fired = player.updatePlayerState(action, self.countdowns[player], self.dt)

            if fired:
                self.countdowns[player] = player.weapon.maxFrameCooldown
            else:
                self.countdowns[player] -= 1

        #Handle collision
        self.handlePlayerCollisions()
        self.handleBlockCollisions()
        
        #Handle ray collisions
        deltaDamageCredit, deltaEliminationCredit, deltaDeathCredit = self.handleRayCollisions(rays)        
        
        #Create rews
        rewards = {a: 0 for a in self.agents}

        #Attribute rewards
        for player in self.players:
            rew = deltaDamageCredit[player] + deltaEliminationCredit[player] * 75 - 75 * deltaDeathCredit[player]
            a = self.invPlayerLabels[player]
            rewards[a] = rew

        #Adjust score
        for player in self.players:
            deltaScore = deltaDamageCredit[player] + deltaEliminationCredit[player] * 75
            self.scores[player.team] += deltaScore

        #Truncation conditions
        truncations = {a: False for a in self.agents}
        if self.timesteps > self.max_timestep:
            truncations = {a: True for a in self.agents}

        #Termination condition
        terminations = {a: False for a in self.agents}
        for i in range(0, 2):
            if self.scores[i] > self.max_score:
                for player in self.players:
                    a = self.invPlayerLabels[player]
                    if player.team == i:
                        rewards[a] += 200
                    else:
                        rewards[a] -= 100

        #Get observations
        observations = {a: self.getObservations(self.playerLabel[a]) for a in self.agents}

        #Additional info - dummy
        infos = {a: {} for a in self.agents}

        #If terminated
        if any(terminations.values()) or all(truncations.values()):
            self.agents = []

        #Increment timestep
        self.timestep += 1

        return None #out
        
    

In [31]:
class MultiDemo(EnvBase):

    def __init__(self, batch_size=(), device=None, **kwargs):
        self.x = 0
        self.y = 0
        self.max_move_d = 2
        self.n_agents = 2

        self._batch_size = batch_size

        self.device = device

    def _make_spec(self, td_params):
        agent_i_observation_spec = UnboundedContinuousTensorSpec(shape=(2,), dtype=torch.float32)

        agent_i_action_spec = BoundedTensorSpec(low=-2,
                              high=2,
                              shape=(2,),
                              dtype=torch.float32)

        agent_i_reward_spec = UnboundedContinuousTensorSpec(shape=(), dtype=torch.float32)

        action_specs = []
        observation_specs = []
        reward_specs = []
        info_specs = []

        for i in range(self.n_agents):
            action_specs.append(agent_i_action_spec)
            observation_specs.append(agent_i_observation_spec)
            reward_specs.append(agent_i_reward_spec)


        self.action_spec = CompositeSpec(
            {
                "agents": CompositeSpec(
                    {"action": torch.stack(action_specs, dim=0)}, shape=(self.n_agents,)
                )
            }
        )

        self.reward_spec = CompositeSpec(
            {
                "agents": CompositeSpec(
                    {"reward": torch.stack(reward_specs, dim=0)}, shape=(self.n_agents,)
                )
            }
        )

        self.observation_spec = CompositeSpec(
            {
                "agents": CompositeSpec(
                    {"observation": torch.stack(observation_specs, dim=0)}, shape=(self.n_agents,)
                )
            }
        )

        self.done_spec = DiscreteTensorSpec(
            n=2,
            shape=torch.Size((1,)),
            dtype=torch.bool,
        )
    
    def _step(self, tensordict):

        #Get actions
        #2 players
        #player1 +ve
        self.x += tensordict["action", "player1"][0].clamp(-self.max_move, self.max_move) - tensordict["action", "player2"][0].clamp(-self.max_move, self.max_move)
        self.y += tensordict["action", "player1"][1].clamp(-self.max_move, self.max_move) - tensordict["action", "player2"][1].clamp(-self.max_move, self.max_move)
        rewards = [self.x + self.y, -self.x-self.y]

        out = TensorDict(
            {
                "agents": None
            }
        )

        return out

    def _reset(self, seed=None):
        pass

    def _set_seed(self, seed=None):
        pass

        
        

In [42]:
from torchrl.envs.libs.vmas import VmasEnv
from torch import multiprocessing

is_fork = multiprocessing.get_start_method() == "fork"
device = (
    torch.device(0)
    if torch.cuda.is_available() and not is_fork
    else torch.device("cpu")
)
vmas_device = device


env = VmasEnv(
    scenario="football",
    num_envs = 1,
    continuous_actions = True,
    max_steps = 200,
    device = vmas_device,
    #n_agents=4,
    n_blue_agents=2,
    n_red_agents=2,
    #ai_blue_agents=True
    ai_red_agents=False,
)

env.full_action_spec
#env.full_action_spec["red", "action"]


CompositeSpec(
    agent_blue: CompositeSpec(
        action: BoundedTensorSpec(
            shape=torch.Size([1, 2, 2]),
            space=ContinuousBox(
                low=Tensor(shape=torch.Size([1, 2, 2]), device=cuda:0, dtype=torch.float32, contiguous=True),
                high=Tensor(shape=torch.Size([1, 2, 2]), device=cuda:0, dtype=torch.float32, contiguous=True)),
            device=cuda:0,
            dtype=torch.float32,
            domain=continuous), device=cuda:0, shape=torch.Size([1, 2])),
    agent_red: CompositeSpec(
        action: BoundedTensorSpec(
            shape=torch.Size([1, 2, 2]),
            space=ContinuousBox(
                low=Tensor(shape=torch.Size([1, 2, 2]), device=cuda:0, dtype=torch.float32, contiguous=True),
                high=Tensor(shape=torch.Size([1, 2, 2]), device=cuda:0, dtype=torch.float32, contiguous=True)),
            device=cuda:0,
            dtype=torch.float32,
            domain=continuous), device=cuda:0, shape=torch.Size([1, 

In [38]:
env = MultiDemo()

In [39]:
env._make_spec(None)

In [40]:
env.action_keys
env.full_action_spec

CompositeSpec(
    agents: CompositeSpec(
        action: BoundedTensorSpec(
            shape=torch.Size([2, 2]),
            space=ContinuousBox(
                low=Tensor(shape=torch.Size([2, 2]), device=cpu, dtype=torch.float32, contiguous=True),
                high=Tensor(shape=torch.Size([2, 2]), device=cpu, dtype=torch.float32, contiguous=True)),
            device=cpu,
            dtype=torch.float32,
            domain=continuous), device=None, shape=torch.Size([2])), device=None, shape=torch.Size([]))

In [18]:
env.full_observation_spec

CompositeSpec(
    agents: CompositeSpec(
        observation: UnboundedContinuousTensorSpec(
            shape=torch.Size([1, 4, 18]),
            space=None,
            device=cuda:0,
            dtype=torch.float32,
            domain=continuous),
        info: CompositeSpec(
            pos_rew: UnboundedContinuousTensorSpec(
                shape=torch.Size([1, 4, 1]),
                space=None,
                device=cuda:0,
                dtype=torch.float32,
                domain=continuous),
            final_rew: UnboundedContinuousTensorSpec(
                shape=torch.Size([1, 4, 1]),
                space=None,
                device=cuda:0,
                dtype=torch.float32,
                domain=continuous),
            agent_collisions: UnboundedContinuousTensorSpec(
                shape=torch.Size([1, 4, 1]),
                space=None,
                device=cuda:0,
                dtype=torch.float32,
                domain=continuous), device=cuda:0, shape=